## A Geo Enabled, Entity Resolved Knowledge Graph

Now we are going to put it all together as a geoERKG.  We will create the graph similar to what we did in notebook `03` where resolved entities become nodes, the relationships Senzing published as match keys become edges, and the placekey and coordinates ride along as properties on each node.  We will then use an LLM to ask questions and observe how the answers have changed from notebook `04` now that they are backed with the geoERKG.

In [ ]:
import collections, json, math, os, re, time

import anthropic
import grpc
from IPython.display import Markdown, display
import networkx as nx
import openai
import pandas as pd
from senzing import SzEngineFlags as F
from senzing_grpc import SzAbstractFactoryGrpc
from sqlalchemy import create_engine

channel = grpc.insecure_channel(
    f"{os.getenv('SENZING_GRPC_HOST', 'resolver')}:8261",
    options=[('grpc.max_receive_message_length', 200 * 1024 * 1024)])
grpc.channel_ready_future(channel).result(timeout=20)
engine = SzAbstractFactoryGrpc(channel).create_engine()
print("✅ resolver connected")

db = create_engine(
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'postgres')}:"
    f"{os.getenv('POSTGRES_PASSWORD', 'workshop')}@"
    f"{os.getenv('POSTGRES_HOST', 'postgres')}:5432/{os.getenv('POSTGRES_DB', 'erkg')}")
print("✅ database connected")

HYATT = 2120          # the conference hotel - every question starts here

## Geocodes as node properties

We keep every address an entity has, not just one.  This is because, unlike in notebook 04 where the only thing we could do was deal with raw rows that each contained one address, we now have the ability to create a holistic view of an entity.  In that view, entities can have multiple addresses, which you will see makes for a more accurate RAG experience.  

In [ ]:
geo = pd.read_sql("""
    SELECT entity_id, street, city, placekey, latitude, longitude
    FROM workshop.entity_geo
    WHERE placekey IS NOT NULL AND street NOT LIKE 'PO BOX%%'
""", db)

addresses = collections.defaultdict(list)
for r in geo.itertuples():
    addresses[int(r.entity_id)].append(
        {'street': r.street, 'city': r.city, 'placekey': r.placekey,
         'lat': r.latitude, 'lon': r.longitude})

print(f"{len(addresses):,} entities have a placekey")

## Building the graph

Every resolved entity becomes a node, including the ones we could not geocode.  The node's properties its name, address(es), the source(s) it came from, the records that merged into it, and its Placekey and coordinates when we have them.  The relationships are all the of the Senzing match keys, which are added in the next cells.

Note that this might take up to 1 minute to run.

In [ ]:
t0 = time.time()
G = nx.Graph()
pending_links = []

handle = engine.export_json_entity_report(F.SZ_EXPORT_DEFAULT_FLAGS)
try:
    while (line := engine.fetch_next(handle)):
        d = json.loads(line)
        e = d['RESOLVED_ENTITY']
        eid = e['ENTITY_ID']
        recs = e.get('RECORDS', [])
        a = addresses.get(eid, [])

        G.add_node(eid,
                   name=e.get('ENTITY_NAME') or f'Entity {eid}',
                   sources=sorted({r['DATA_SOURCE'] for r in recs}),
                   n_records=len(recs),
                   records=[f"{r['DATA_SOURCE']}:{r['RECORD_ID']}" for r in recs[:8]],
                   addresses=a,
                   placekey=a[0]['placekey'] if a else None,
                   street=a[0]['street'] if a else None,
                   city=a[0]['city'] if a else None,
                   lat=a[0]['lat'] if a else None,
                   lon=a[0]['lon'] if a else None)

        for r in d.get('RELATED_ENTITIES', []):
            pending_links.append((eid, r['ENTITY_ID'], r.get('MATCH_KEY', ''),
                                  bool(r.get('IS_AMBIGUOUS', 0))))
finally:
    engine.close_export_report(handle)

print(f"{G.number_of_nodes():,} entity nodes in {time.time() - t0:.0f}s")

## Edges from entity resolution

Senzing reports each relationship bidirectionally, so we add a pair only once.  

In [ ]:
for a, b, match_key, ambiguous in pending_links:
    if a < b and G.has_node(b):         
        G.add_edge(a, b, match_key=match_key, ambiguous=ambiguous)

print(f"{G.number_of_edges():,} entity-resolution edges")

## What we built

Note that not all nodes have a Placekey.  Only the conference neighborhood was geocoded even though the data we have is statewide for some of the sources.  This means that the graph is statewide while the geography is only local for this workshop.

In [ ]:
located = [n for n, a in G.nodes(data=True) if a['lat'] is not None]
keys = collections.Counter(d['match_key'] for _, _, d in G.edges(data=True))

print(f"entities          {G.number_of_nodes():,}")
print(f"  with a placekey {len(located):,}")
print(f"ER edges          {G.number_of_edges():,}")
print(f"  ambiguous       {sum(1 for _,_,d in G.edges(data=True) if d['ambiguous']):,}")
print(f"\nmatch keys: {keys.most_common(6)}")
print(f"\nanchor: {G.nodes[HYATT]['name']} at {G.nodes[HYATT]['street']}")

## Calculating distance from the Placekey

A placekey is written `what@where`. The `where` is a hexagon of known size with an edge length of ~76 m or ~135 m across, so two entities sharing it are likely in the same or neighboring buildings.  Here, we use the Haversine equation to get distance in meters (after converting the Placekey to lat/lon coordiates.

In [ ]:
def cell(placekey):
    """A placekey is what@where. The 'where' is the H3 hexagon."""
    return placekey.split('@')[-1]


def meters(a, b):
    x, y = G.nodes[a], G.nodes[b]
    la1, lo1, la2, lo2 = x['lat'], x['lon'], y['lat'], y['lon']
    return 6371000 * 2 * math.asin(math.sqrt(
        math.sin(math.radians(la2 - la1) / 2) ** 2 +
        math.cos(math.radians(la1)) * math.cos(math.radians(la2)) *
        math.sin(math.radians(lo2 - lo1) / 2) ** 2))


# A sanity check on meters(): two businesses sharing a placekey must read 0,
# because they share a cell centroid. One in a different cell must not.
for other, note in ((1259, 'same placekey as the hotel'),
                    (2783, "MAX'S, a few hundred meters away")):
    print(f"{G.nodes[HYATT]['name'][:28]} -> {G.nodes[other]['name'][:28]}: "
          f"{meters(HYATT, other):>5.0f} m   ({note})")

## Gathering the neighborhood

This is our retrieval step in this notebook, which is every entity within a given distance of the anchor, grouped by the H3 cell its placekey falls in.

Grouping is on the `where` half of the placekey, not the whole thing, so suites and separate addresses that share a hexagon collapse into one group.  

In [ ]:
def neighborhood(anchor=HYATT, radius_m=600, max_places=25):
    """Every located entity within radius_m of the anchor, grouped by H3 cell."""
    anchor_cell = cell(G.nodes[anchor]['placekey'])
    near = sorted(((n, meters(anchor, n)) for n in located if n != anchor),
                  key=lambda t: t[1])

    by_place = collections.OrderedDict()
    for n, d in near:
        c = cell(G.nodes[n]['placekey'])
        by_place.setdefault(c, {'d': d, 'ents': []})
        if d <= radius_m or c == anchor_cell:
            by_place[c]['ents'].append(n)

    places = {c: v for c, v in by_place.items() if v['ents']}
    places = dict(list(places.items())[:max_places])
    members = {anchor} | {n for v in places.values() for n in v['ents']}
    return places, members


places, members = neighborhood()
print(f"{len(places)} locations, {len(members)} entities")

## Turning the neighborhood into context

One function builds everything the model sees: who is in the hotel, what is nearby and how far, which entities resolution linked and on what evidence, which links reach outside the neighborhood, and which addresses look unreliable.  There is no separate code path per question, which is what lets one function answer a wide range of them.

In [ ]:
def describe(anchor=HYATT, radius_m=600, max_places=200):
    places, members = neighborhood(anchor, radius_m, max_places)
    A = G.nodes[anchor]

    out = [f"STARTING POINT: {A['name']} (entity {anchor})",
           f"  {A['street']}, {A['city']} | placekey {A['placekey']} "
           f"| sources: {', '.join(A['sources'])}",
           f"  source records merged into this one entity: {', '.join(A['records'])}",
           "",
           "SURROUNDING LOCATIONS - meters from the starting point, computed from",
           "Placekey coordinates.",
           ""]

    for c, v in places.items():
        streets = sorted({a['street'] for n in v['ents'] for a in G.nodes[n]['addresses']
                          if cell(a['placekey']) == c})
        out.append(f"{v['d']:.0f} m - {streets[0]}  [cell {c}]")
        for n in v['ents'][:12]:
            a = G.nodes[n]
            out.append(f"    {n} {a['name']} | {', '.join(a['sources'])} "
                       f"| {a['n_records']} record(s)")
        if len(v['ents']) > 12:
            out.append(f"    ... and {len(v['ents']) - 12} more")

    out += ["", "ENTITY RESOLUTION LINKS. Senzing decided these are related; the key",
            "says why. +ADDRESS alone means only a shared address. +NAME or +PHONE",
            "means the match would survive the business moving premises."]

    inner, outer = [], []
    for m in sorted(members):
        for nb in G.neighbors(m):
            if nb < m and nb in members:
                continue
            (inner if nb in members else outer).append((m, nb, G.edges[m, nb]))

    # Most links are +ADDRESS, which is near-tautological with sharing a placekey.
    # Sort the address-independent ones first so truncation cannot discard them.
    def interest(t):
        d = t[2]
        return (0 if d['ambiguous'] else 1,
                0 if ('+NAME' in d['match_key'] or '+PHONE' in d['match_key']) else 1)

    inner.sort(key=interest)
    outer.sort(key=interest)

    def fmt(m, nb, d):
        return (f"  {m} {G.nodes[m]['name'][:32]} <-> {nb} {G.nodes[nb]['name'][:32]}"
                f"  {d['match_key']}{' (AMBIGUOUS)' if d['ambiguous'] else ''}")

    out.append("  -- between businesses in this neighborhood:")
    out += [fmt(m, nb, d) for m, nb, d in inner[:50]]

    out.append("  -- reaching OUTSIDE this neighborhood:")
    for m, nb, d in outer[:30]:
        b = G.nodes[nb]
        if b['lat'] is not None:
            out.append(fmt(m, nb, d) +
                       f"  [{b['street']}, {meters(m, nb):.0f} m from here]")
        else:
            out.append(fmt(m, nb, d) + "  [no address on file]")

    out += ["", "DATA QUALITY. A Placekey is what@where. A long 'what' beginning with 0",
            "means the address could not be matched, so its coordinates are unreliable."]
    for n in sorted(members):
        a = G.nodes[n]
        cells = {cell(x['placekey']) for x in a['addresses']}
        if len(cells) > 1:
            out.append(f"  entity {n} {a['name']} has {len(a['addresses'])} addresses "
                       f"in {len(cells)} different cells:")
            out += [f"      {x['street']} -> {x['placekey']}" for x in a['addresses']]

    return "\n".join(out)

### A quick check that we are getting what we expect in the nearby neighborhood...

In [ ]:
ctx = describe()
print(f"{len(ctx):,} characters (~{len(ctx)//4:,} tokens)\n")
print(ctx[:1400])

## Letting the model choose the search radius

It is possible, or even likely, that the question from the user will contain some information about the distance they are interested in.  So rather than hard coding a search radius, we create a planner whose purpose is simply to understand and extract distances from the user's input using a small, cheap LLM.  We then provide the LLM some instructions on what is meant if the user says something like "walking distance," "blocks," or "a 10 minute walk."

**Important Note**

Remember that distance is being measured by GPS coordinates and not a map of city blocks.  So the actual walking distance is likely longer than this.  This is done for coding simplicity and you can always make this more accurate, but that is beyond the scope of this workshop.

In [ ]:
anthropic_client = anthropic.Anthropic() if os.getenv("ANTHROPIC_API_KEY") else None
openai_client    = openai.OpenAI()       if os.getenv("OPENAI_API_KEY")    else None

ANSWER_MODEL  = {"anthropic": "claude-opus-5",    "openai": "gpt-5.4-nano"}
PLANNER_MODEL = {"anthropic": "claude-haiku-5-5", "openai": "gpt-5.4-nano"}

print("Anthropic:", anthropic_client is not None, " OpenAI:", openai_client is not None)
print("answering with:", ANSWER_MODEL)
print("planning with: ", PLANNER_MODEL)

In [ ]:
PLANNER_PROMPT = """You read a search radius, in meters, out of a question about businesses 
    near a hotel. Reply with ONLY an integer, or ONLY the word NONE.

    If the question states a distance, time or count, convert it exactly:
      '1000m' -> 1000       '300 m' -> 300        '2 km' -> 2000
      'half a mile' -> 805  'a mile' -> 1609      '3 blocks' -> 360
      '10 minute walk' -> 800
      (a block is 120 m; walking is 80 m per minute)

    If it states no number but describes a distance, use:
      'the same building', 'the same address' -> 1
      'next door', 'two doors down', 'immediately outside' -> 250
      'near', 'nearby', 'close by', 'around here' -> 600
      'walk', 'walking distance', 'on foot' -> 800
      'this neighborhood', 'the area', 'the wider area' -> 1500
      'anywhere', 'in Burlingame', 'the whole town' -> 3000

    Any mention of a building, address, premises, walking or proximity IS a 
    distance - map it above, never NONE.

    Reply NONE only when the question refers to no place or distance at all - 
    for example about data quality, which records merged, or how businesses 
    are connected to each other.

    Worked examples:
      Q: What else operates out of the same building as the hotel?  A: 1
      Q: Who shares this address?  A: 1
      Q: Anywhere I could walk to for a drink?  A: 800
      Q: What are my options within 1000m?  A: 1000
      Q: Which links is the system unsure about?  A: NONE
      Q: What records merged into the hotel entity?  A: NONE

    Reply with a single integer between 1 and 5000, or NONE. Nothing else.
"""


def radius_for(question, provider="anthropic"):
    """Read the distance out of the question using the small planner model.
    Returns None when the question contains no distance at all, so there is no
    fallback radius hidden in here."""
    if provider == "anthropic":
        r = anthropic_client.messages.create(
            model=PLANNER_MODEL["anthropic"], max_tokens=2000,
            system=PLANNER_PROMPT,
            messages=[{"role": "user", "content": question}])
        text = "".join(b.text for b in r.content if b.type == "text").strip()
    else:
        r = openai_client.chat.completions.create(
            model=PLANNER_MODEL["openai"], max_completion_tokens=2000,
            messages=[{"role": "system", "content": PLANNER_PROMPT},
                      {"role": "user", "content": question}])
        text = (r.choices[0].message.content or "").strip()

    if re.search(r'\bNONE\b', text, re.I):
        return None
    m = re.search(r'\d+', text)
    return max(1, min(int(m.group()), 5000)) if m else None


for q in ["What are my options within 1000m?",
          "What else operates out of the same building as the hotel?",
          "Anywhere I could walk to for a drink?",
          "Which links is the system unsure about?"]:
    print(f"  {str(radius_for(q)):>5}   {q}")

## The system prompt

Now we describe the complete process for the LLM, including the geographic information as well as the output of Senzing.

In [ ]:
SYSTEM_PROMPT = """You answer questions about businesses around a hotel, using a knowledge 
    graph built from California public records. The records were entity-resolved 
    by Senzing - one entity is one real business, built from several source 
    records - and geocoded with Placekey. 
    
    DISTANCES ARE COMPUTED, NOT GUESSED: they are straight-line meters from 
    Placekey coordinates, so you may quote them and compare them. Treat 400 m as 
    a short walk, 800 m as about ten minutes. 
    
    Distances are measured between the centers of Placekey cells, which are 
    hexagons about 135 m across. So two businesses reading 0 m apart share a 
    cell and are on the same block, which often but not always means the same 
    building. Any distance under about 150 m is approximate, and you should say 
    so rather than implying precision you do not have. Comparisons between 
    distances are reliable even when the absolute figures are rough.
    
    ABC-RETAIL and ABC-NONRETAIL mean a liquor license; no other source implies 
    alcohol. 
    
    Links marked +ADDRESS, +NAME or +PHONE were made by entity resolution - say 
    which key supports a claim. AMBIGUOUS means Senzing is unsure. 
    Answer only from the context, cite entity IDs, and say plainly when the 
    context does not support an answer.
"""

## The full QA bot, backed with the geoERKG

This is the function that handles the complete question answering.  It first uses the cheap LLM to extract distance.  It then uses the context obtained from the geoERKG to answer the user's question.  

In [ ]:
def ask(question, provider="anthropic", max_tokens=8000):
    """Ask anything about the hotel's surroundings. The distance comes from the
    question; if the question has none, the search area is the whole dataset."""
    radius_m = radius_for(question, provider)

    if radius_m is None:
        # No distance in the question, so do not invent one - search everything
        # that has coordinates and say so.
        radius_m = 5000
        scope = "no distance given, so searching every geocoded business"
    else:
        scope = "from the question"

    context = describe(radius_m=radius_m)

    print(f"Question: {question}")
    print(f"Searching within {radius_m} m ({scope})")
    print(f"Context: {len(context):,} chars (~{len(context)//4:,} tokens)")

    msg = f"Knowledge graph context:\n{context}\n\nQuestion: {question}"

    if provider == "anthropic":
        r = anthropic_client.messages.create(
            model=ANSWER_MODEL["anthropic"], max_tokens=max_tokens,
            system=SYSTEM_PROMPT, messages=[{"role": "user", "content": msg}])
        answer = "".join(b.text for b in r.content if b.type == "text")
        if not answer:
            print(f"⚠️  no text returned. stop_reason={r.stop_reason}, "
                  f"blocks={[b.type for b in r.content]}, "
                  f"output_tokens={r.usage.output_tokens}")
            return
    else:
        r = openai_client.chat.completions.create(
            model=ANSWER_MODEL["openai"], max_completion_tokens=max_tokens,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": msg}])
        answer = r.choices[0].message.content
        if not answer:
            print(f"⚠️  no text returned. finish_reason={r.choices[0].finish_reason}")
            return

    # The model answers in markdown, so render it rather than printing it raw.
    display(Markdown(answer))

## Questions

Experiment with this section with questions of your own.  Here are some good ones to try:

- I am staying at this hotel.  What else operates out of the same building?
- Where's the nearest place I could get a drink, and how far is the walk?
- Which businesses around the hotel are connected to each other, and what evidence connects them?
- Are any of the businesses near the hotel part of something larger elsewhere in California?
- Is anything in this data wrong or suspicious?
- Find a bar not in a hotel that is within 1000 m.
- Tell me about Top Golf and can I get a drink there?

In [ ]:
ask("I'm staying at this hotel. What else operates out of the same building?")